# Kaggle 性別預測模型建立完整流程
modelBertRf20

In [ ]:
# !pip install imbalanced-learn missingpy xgboost catboost optuna sentence-transformers -q

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.combine import SMOTEENN
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

## 第一步：準備資料與預防資料洩漏
讀取資料並先進行 Train/Test Split (抽出驗證集)。

In [2]:
train_path = 'data/boy or girl 2025 train_missingValue.csv'
test_path = 'data/boy or girl 2025 test no ans_missingValue.csv'

df_train_full = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

X_full = df_train_full.drop(columns=['gender'])
y_full = df_train_full['gender']

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)
print('Train shape:', X_train.shape)
print('Valid shape:', X_valid.shape)
print('Test shape:', df_test.shape)

Train shape: (338, 10)
Valid shape: (85, 10)
Test shape: (426, 11)


## Phase 0, 1, 2: 資料清理、異常值處理與特徵萃取

In [3]:
def is_repeated_or_symbol(s):
    if pd.isna(s) or s == '' or s == 'nan': return False
    if re.fullmatch(r'[^A-Za-z0-9\u4e00-\u9fff]+', s): return True
    if re.fullmatch(r'(.)\1{3,}', s): return True
    return False

def build_features(df):
    df = df.copy()
    
    # Phase 0: yt & phone_os
    if 'yt' in df.columns:
        orig_yt = df['yt']
        df['yt'] = pd.to_numeric(df['yt'], errors='coerce')
        df['is_yt_invalid'] = np.where(orig_yt.notna() & df['yt'].isna(), 1, 0)
    
    if 'phone_os' in df.columns:
        df['phone_os_clean'] = df['phone_os'].astype(str).str.strip().str.lower().replace({'iphone': 'apple'})
        valid_os = {'android', 'apple', 'windows'}
        df['is_phone_os_invalid'] = np.where(df['phone_os_clean'].isin(valid_os), 0, 1)
        df.loc[df['is_phone_os_invalid'] == 1, 'phone_os_clean'] = 'Unknown'
        df.drop(columns=['phone_os'], inplace=True, errors='ignore')

    # Phase 1: 異常值標記與截斷
    if 'weight' in df.columns:
        df['is_weight_missing'] = df['weight'].isna().astype(int)
        df['is_weight_outlier'] = np.where(df['weight'].notna() & ((df['weight'] < 30) | (df['weight'] > 1000)), 1, 0)
        df['weight'] = df['weight'].clip(lower=40, upper=120)

    if 'height' in df.columns:
        df['is_height_missing'] = df['height'].isna().astype(int)
        df['is_height_outlier'] = np.where(df['height'].notna() & ((df['height'] < 130) | (df['height'] > 220)), 1, 0)
        df['height'] = df['height'].clip(lower=140, upper=200)

    if 'iq' in df.columns:
        df['is_iq_outlier'] = np.where(df['iq'].notna() & ((df['iq'] < 100) | (df['iq'] > 140)), 1, 0)
        df['iq'] = df['iq'].clip(lower=100, upper=140)

    if 'fb_friends' in df.columns:
        df['is_fb_friends_outlier'] = np.where(df['fb_friends'].notna() & ((df['fb_friends'] < 0) | (df['fb_friends'] > 2000)), 1, 0)
        df['fb_friends'] = df['fb_friends'].clip(lower=0, upper=2000)

    if 'yt' in df.columns:
        df['is_yt_outlier'] = np.where(df['yt'].notna() & ((df['yt'] < 0) | (df['yt'] > 24)), 1, df.get('is_yt_invalid', 0))
        df['yt'] = df['yt'].clip(lower=0, upper=24)

    if 'star_sign' in df.columns:
        # print(df['star_sign'])
        star_map = {
            '牡羊座': 'aries', '金牛座': 'taurus', '雙子座': 'gemini', '巨蟹座': 'cancer',
            '獅子座': 'leo', '處女座': 'virgo', '天秤座': 'libra', '天蠍座': 'scorpio',
            '射手座': 'sagittarius', '摩羯座': 'capricorn', '水瓶座': 'aquarius', '雙魚座': 'pisces'
        }

        if 'star_sign' in df.columns:
            # 原始清洗
            df['star_sign_clean'] = df['star_sign'].astype(str).str.strip()
            # print(df['star_sign'])
            # 判斷是否無效 (檢查中文是否存在於 map 的 keys 中)
            df['is_star_sign_invalid'] = np.where(
                df['star_sign_clean'].isin(star_map.keys()), 
                0, 1
            )
            
            # 執行翻譯：在 map 裡的轉英文，不在裡面的轉 'Unknown'
            df['star_sign_clean'] = df['star_sign_clean'].map(star_map).fillna('Unknown')
            
            df.drop(columns=['star_sign'], inplace=True, errors='ignore')


    # Phase 2: 自我介紹文本特徵 (保留 self_intro 給後續編碼)
    if 'self_intro' in df.columns:
        df['is_intro_missing'] = df['self_intro'].isna().astype(int)
        df['intro_char_length'] = df['self_intro'].fillna('').astype(str).str.len()
        df['intro_word_count'] = df['self_intro'].fillna('').astype(str).str.split().apply(len)
        df['intro_is_all_lower'] = df['self_intro'].fillna('').astype(str).apply(lambda s: s.islower() if s else False).astype(int)
        df['intro_is_all_upper'] = df['self_intro'].fillna('').astype(str).apply(lambda s: s.isupper() if s else False).astype(int)

        df['is_intro_anomalous'] = 0
        df.loc[df['intro_char_length'] == 0, 'is_intro_anomalous'] = 1
        df.loc[df['intro_char_length'] > 500, 'is_intro_anomalous'] = 1
        df.loc[df['self_intro'].fillna('').astype(str).apply(is_repeated_or_symbol), 'is_intro_anomalous'] = 1
        
        # 僅把異常文字設為 NaN，不在這裡 drop，讓後續可以做 tfidf/bert 編碼
        df.loc[df['is_intro_anomalous'] == 1, 'self_intro'] = np.nan
        
    return df

X_train_clean = build_features(X_train)
X_valid_clean = build_features(X_valid)
X_test_clean = build_features(df_test)

print("特徵工程完成")

# test = build_features(X_train[X_train['id'] == 163])

特徵工程完成


In [ ]:
# test

In [4]:
pd.set_option('display.max_columns', None)
print(X_train_clean.head(20))

      id  height  weight  sleepiness     iq  fb_friends      yt  \
145  146   168.0    63.0         2.0  100.0       328.0   2.000   
78    79   170.0     NaN         NaN  100.0        99.0  10.000   
16    17   156.0    47.0         NaN  130.0       400.0   3.500   
149  150   168.0    49.0         1.0  100.0      2000.0  24.000   
293  294     NaN    76.0         4.0  140.0      1287.0  24.000   
94    95     NaN    74.0         NaN  140.0      2000.0   2.000   
250  251   171.0    59.0         3.0  131.0      1234.0   1.000   
333  334   178.0     NaN         3.0  140.0      1000.0   4.000   
332  333   183.0    79.0         5.0    NaN       237.0  19.000   
21    22   165.0    50.0         4.0  115.0       209.0   9.590   
373  374     NaN    55.0         3.0  120.0       555.0   6.000   
194  195   181.0     NaN         NaN  140.0       100.0     NaN   
13    14     NaN    53.0         4.0    NaN       300.0   2.500   
218  219   180.0    70.0         4.0  100.0       100.0  24.00

## Phase 2.5: self_intro 編碼 (BERT Embedding + Intent Clustering)
可切換 `SELF_INFO_ENCODER = 'tfidf'` 或 `'bert'`。
**新增**：提取 self_intro 意圖類別（KMeans），減低維度同時保留語義。
注意：向量器只在訓練集 fit，驗證與測試只做 transform，避免資料洩漏。

In [5]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

# ====== 設定參數 ======
SELF_INFO_ENCODER = 'bert'  # 'bert' or 'tfidf'
SELF_INFO_BERT_PCA_COMPONENTS = 24  # 0 表示不壓縮

def _has_intro_mask(df):
    """可用於文本語意建模的樣本遮罩：非缺失且非空白。"""
    text_non_empty = df['self_intro'].fillna('').astype(str).str.strip().ne('')
    if 'is_intro_missing' in df.columns:
        return df['is_intro_missing'].fillna(0).astype(int).eq(0) & text_non_empty
    return text_non_empty

def extract_gender_prototype_features(train_df, train_labels, valid_df, test_df, model):
    """
    只在 train 上計算 male/female centroid，並計算各集合到 centroid 的相似度。
    假設 labels 已經是數值：1=boy, 2=girl
    重要：centroid 只使用 has_intro=True 的樣本。
    """
    train_text = train_df['self_intro'].fillna('').astype(str).tolist()
    train_emb = model.encode(train_text, convert_to_numpy=True, normalize_embeddings=True)

    # 先依標籤分，再排除無自介內容樣本
    train_has_intro = _has_intro_mask(train_df)
    male_mask = (train_labels == 1) & train_has_intro
    female_mask = (train_labels == 2) & train_has_intro

    # 若某一類可用樣本為 0，退回只用標籤（避免崩潰）
    if male_mask.sum() == 0:
        male_mask = (train_labels == 1)
    if female_mask.sum() == 0:
        female_mask = (train_labels == 2)

    male_centroid = train_emb[male_mask].mean(axis=0)
    female_centroid = train_emb[female_mask].mean(axis=0)

    print(f"   [Prototype] 計算完成 - 男性樣本: {male_mask.sum()}, 女性: {female_mask.sum()}")

    def get_sim_df(df, original_emb=None):
        if original_emb is None:
            text = df['self_intro'].fillna('').astype(str).tolist()
            emb = model.encode(text, convert_to_numpy=True, normalize_embeddings=True)
        else:
            emb = original_emb

        sim_m = emb @ male_centroid
        sim_f = emb @ female_centroid

        # 缺失/空白自介給中性值，避免空字串語意向量引入偏差
        has_intro = _has_intro_mask(df).to_numpy()
        sim_m = np.where(has_intro, sim_m, 0.0)
        sim_f = np.where(has_intro, sim_f, 0.0)

        return pd.DataFrame({
            'self_intro_sim_to_male': sim_m,
            'self_intro_sim_to_female': sim_f,
            'self_intro_male_minus_female': sim_m - sim_f
        }, index=df.index)

    train_proto = get_sim_df(train_df, original_emb=train_emb)
    valid_proto = get_sim_df(valid_df)
    test_proto = get_sim_df(test_df)

    return train_proto, valid_proto, test_proto

def encode_self_intro(train_df, train_labels, valid_df, test_df,
                      method='bert', pca_components=0, use_gender_prototype=True):
    """
    pca_components 選項：
    > 0  : 執行 PCA 壓縮並保留特徵
    == 0 : 保留原始 BERT 全維度 (384維)
    == -1: 僅保留性別原型相似度特徵，不保留任何 BERT Embedding 向量
    """
    train_df, valid_df, test_df = train_df.copy(), valid_df.copy(), test_df.copy()
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

    # ====== 1. 提取性別原型特徵 ======
    empty_train = pd.DataFrame(index=train_df.index)
    empty_valid = pd.DataFrame(index=valid_df.index)
    empty_test = pd.DataFrame(index=test_df.index)
    proto_features = [empty_train, empty_valid, empty_test]

    if use_gender_prototype:
        print("   提取性別原型相似度特徵...")
        train_proto, valid_proto, test_proto = extract_gender_prototype_features(
            train_df, train_labels, valid_df, test_df, model
        )
        proto_features = [train_proto, valid_proto, test_proto]

    # ====== 2. 處理 BERT Embedding (判斷是否保留) ======
    extra_features = []
    if pca_components != -1:
        print(f"   執行 {method.upper()} 編碼 (保留 Embedding)...")
        train_text = train_df['self_intro'].fillna('').astype(str).tolist()
        valid_text = valid_df['self_intro'].fillna('').astype(str).tolist()
        test_text = test_df['self_intro'].fillna('').astype(str).tolist()

        train_emb = model.encode(train_text, convert_to_numpy=True, normalize_embeddings=True)
        valid_emb = model.encode(valid_text, convert_to_numpy=True, normalize_embeddings=True)
        test_emb = model.encode(test_text, convert_to_numpy=True, normalize_embeddings=True)

        if pca_components > 0:
            print(f"   PCA 壓縮至 {pca_components} 維...")
            pca = PCA(n_components=pca_components, random_state=42)
            train_emb = pca.fit_transform(train_emb)
            valid_emb = pca.transform(valid_emb)
            test_emb = pca.transform(test_emb)

        feat_names = [f'bert_feat_{i}' for i in range(train_emb.shape[1])]
        extra_features = [
            pd.DataFrame(train_emb, columns=feat_names, index=train_df.index),
            pd.DataFrame(valid_emb, columns=feat_names, index=valid_df.index),
            pd.DataFrame(test_emb, columns=feat_names, index=test_df.index)
        ]
    else:
        print("   跳過 BERT Embedding 保留，僅使用原型相似度特徵。")

    # ====== 3. 最終合併與清理 ======
    def finalize_df(df, proto_df, extra_df=None):
        if 'self_intro' in df.columns:
            df = df.drop(columns=['self_intro'])

        to_concat = [df]
        if proto_df is not None and not proto_df.empty:
            to_concat.append(proto_df)
        if extra_df is not None:
            to_concat.append(extra_df)

        return pd.concat(to_concat, axis=1)

    train_extra_df = extra_features[0] if extra_features else None
    valid_extra_df = extra_features[1] if extra_features else None
    test_extra_df = extra_features[2] if extra_features else None

    train_df = finalize_df(train_df, proto_features[0], train_extra_df)
    valid_df = finalize_df(valid_df, proto_features[1], valid_extra_df)
    test_df = finalize_df(test_df, proto_features[2], test_extra_df)

    return train_df, valid_df, test_df

# ===== 執行 =====
SELF_INFO_BERT_PCA_COMPONENTS = -1

X_train_clean, X_valid_clean, X_test_clean = encode_self_intro(
    X_train_clean, y_train, X_valid_clean, X_test_clean,
    method=SELF_INFO_ENCODER,
    pca_components=SELF_INFO_BERT_PCA_COMPONENTS,
    use_gender_prototype=True
)

print("\n特徵工程完成！")
print(f"最終維度: {X_train_clean.shape}")
print(f"現有欄位: {X_train_clean.columns.tolist()}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1967.55it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   提取性別原型相似度特徵...
   [Prototype] 計算完成 - 男性樣本: 188, 女性: 67
   跳過 BERT Embedding 保留，僅使用原型相似度特徵。

特徵工程完成！
最終維度: (338, 28)
現有欄位: ['id', 'height', 'weight', 'sleepiness', 'iq', 'fb_friends', 'yt', 'is_yt_invalid', 'phone_os_clean', 'is_phone_os_invalid', 'is_weight_missing', 'is_weight_outlier', 'is_height_missing', 'is_height_outlier', 'is_iq_outlier', 'is_fb_friends_outlier', 'is_yt_outlier', 'star_sign_clean', 'is_star_sign_invalid', 'is_intro_missing', 'intro_char_length', 'intro_word_count', 'intro_is_all_lower', 'intro_is_all_upper', 'is_intro_anomalous', 'self_intro_sim_to_male', 'self_intro_sim_to_female', 'self_intro_male_minus_female']


In [7]:
target_ids = [7, 10, 31, 51]
result = X_train_clean[X_train_clean['id'].isin(target_ids)]
result

,id,height,weight,sleepiness,iq,fb_friends,yt,is_yt_invalid,phone_os_clean,is_phone_os_invalid,is_weight_missing,is_weight_outlier,is_height_missing,is_height_outlier,is_iq_outlier,is_fb_friends_outlier,is_yt_outlier,star_sign_clean,is_star_sign_invalid,is_intro_missing,intro_char_length,intro_word_count,intro_is_all_lower,intro_is_all_upper,is_intro_anomalous,self_intro_sim_to_male,self_intro_sim_to_female,self_intro_male_minus_female
50,51,174.0,65.0,2.0,100.0,399.0,NaN,0,Unknown,1,0,0,0,0,1,0,0,aquarius,0,0,8,1,1,0,0,0.329676,0.334388,-0.004711
6,7,NaN,50.0,4.0,115.0,209.0,9.59,0,android,0,0,0,1,0,0,0,0,Unknown,1,1,0,0,0,0,1,0.000000,0.000000,0.000000
30,31,175.0,75.0,5.0,100.0,200.0,1.00,0,android,0,0,0,0,0,1,0,0,taurus,0,0,16,4,0,0,0,0.277262,0.303010,-0.025749
9,10,165.0,55.0,5.0,130.0,NaN,2.00,0,apple,0,0,0,0,0,0,0,0,aries,0,0,9,1,0,0,0,0.340853,0.385886,-0.045033


## Phase 3: MissForest 缺失值填補與類別特徵處理
利用 IterativeImputer (基於 Random Forest，此為 sklearn 原生穩定版本，表現等價於 missingpy 的 MissForest) 對數值補值。

In [8]:
# 分離數值與類別特徵 (需先確認欄位存在)
num_cols = [c for c in ['height', 'weight', 'iq', 'fb_friends', 'yt', 'sleepiness'] if c in X_train_clean.columns]
cat_cols = [c for c in ['phone_os_clean', 'star_sign_clean'] if c in X_train_clean.columns]

# 類別變數填補 Mode 或 'Unknown'
for c in cat_cols:
    mode = X_train_clean[c].mode()[0] if len(X_train_clean[c].mode()) > 0 else 'Unknown'
    X_train_clean[c].fillna(mode, inplace=True)
    X_valid_clean[c].fillna(mode, inplace=True)
    X_test_clean[c].fillna(mode, inplace=True)

# 數值使用 MissForest (IterativeImputer with RandomForest)
# 備註：為避免 missingpy 與最新版本 sklearn 相容性問題，這裡採用等價的 sklearn API 開發：
rf_estimator = RandomForestRegressor(n_estimators=50, random_state=42)
miss_forest = IterativeImputer(estimator=rf_estimator, random_state=42, max_iter=10)

if len(num_cols) > 0:
    X_train_clean[num_cols] = miss_forest.fit_transform(X_train_clean[num_cols])
    X_valid_clean[num_cols] = miss_forest.transform(X_valid_clean[num_cols])
    X_test_clean[num_cols]  = miss_forest.transform(X_test_clean[num_cols])


X_train_clean.head(5)


,id,height,weight,sleepiness,iq,fb_friends,yt,is_yt_invalid,phone_os_clean,is_phone_os_invalid,is_weight_missing,is_weight_outlier,is_height_missing,is_height_outlier,is_iq_outlier,is_fb_friends_outlier,is_yt_outlier,star_sign_clean,is_star_sign_invalid,is_intro_missing,intro_char_length,intro_word_count,intro_is_all_lower,intro_is_all_upper,is_intro_anomalous,self_intro_sim_to_male,self_intro_sim_to_female,self_intro_male_minus_female
145,146,168.00,63.00,2.00,100.0,328.0,2.0,0,android,0,0,0,0,0,1,0,0,scorpio,0,1,0,0,0,0,1,0.000000,0.000000,0.000000
78,79,170.00,59.68,3.30,100.0,99.0,10.0,0,android,0,1,0,0,0,1,0,0,leo,0,0,4,1,1,0,0,0.316101,0.311759,0.004342
16,17,156.00,47.00,2.72,130.0,400.0,3.5,0,apple,0,0,0,0,0,0,0,0,virgo,0,0,26,5,0,0,0,0.274845,0.322193,-0.047348
149,150,168.00,49.00,1.00,100.0,2000.0,24.0,0,Unknown,1,0,0,0,0,1,0,1,leo,0,0,2,1,0,0,0,0.449749,0.474628,-0.024879
293,294,180.58,76.00,4.00,140.0,1287.0,24.0,0,apple,0,0,0,1,0,1,0,1,Unknown,1,0,17,4,0,0,0,0.250873,0.253375,-0.002502


In [9]:
X_train_clean.columns

Index(['id', 'height', 'weight', 'sleepiness', 'iq', 'fb_friends', 'yt',
       'is_yt_invalid', 'phone_os_clean', 'is_phone_os_invalid',
       'is_weight_missing', 'is_weight_outlier', 'is_height_missing',
       'is_height_outlier', 'is_iq_outlier', 'is_fb_friends_outlier',
       'is_yt_outlier', 'star_sign_clean', 'is_star_sign_invalid',
       'is_intro_missing', 'intro_char_length', 'intro_word_count',
       'intro_is_all_lower', 'intro_is_all_upper', 'is_intro_anomalous',
       'self_intro_sim_to_male', 'self_intro_sim_to_female',
       'self_intro_male_minus_female'],
      dtype='object')

In [10]:
# 對類別變數進行 One-Hot Encode
X_train_enc = pd.get_dummies(X_train_clean, columns=cat_cols, dummy_na=False, dtype=int)
X_valid_enc = pd.get_dummies(X_valid_clean, columns=cat_cols, dummy_na=False, dtype=int)
X_test_enc  = pd.get_dummies(X_test_clean, columns=cat_cols, dummy_na=False, dtype=int)

# 對齊欄位 (訓練集與測試集 One-hot 欄位同步)
X_train_enc, X_valid_enc = X_train_enc.align(X_valid_enc, join='left', axis=1, fill_value=0)
X_train_enc, X_test_enc  = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

# 保留並移除 id 欄位 (不再納入訓練)
train_ids = X_train_enc.pop('id') if 'id' in X_train_enc.columns else None
valid_ids = X_valid_enc.pop('id') if 'id' in X_valid_enc.columns else None
test_ids  = X_test_enc.pop('id') if 'id' in X_test_enc.columns else None


print("缺失值填補與特徵編碼完成，特徵數:", X_train_enc.shape[1])

缺失值填補與特徵編碼完成，特徵數: 41


In [11]:
X_train_enc.columns

Index(['height', 'weight', 'sleepiness', 'iq', 'fb_friends', 'yt',
       'is_yt_invalid', 'is_phone_os_invalid', 'is_weight_missing',
       'is_weight_outlier', 'is_height_missing', 'is_height_outlier',
       'is_iq_outlier', 'is_fb_friends_outlier', 'is_yt_outlier',
       'is_star_sign_invalid', 'is_intro_missing', 'intro_char_length',
       'intro_word_count', 'intro_is_all_lower', 'intro_is_all_upper',
       'is_intro_anomalous', 'self_intro_sim_to_male',
       'self_intro_sim_to_female', 'self_intro_male_minus_female',
       'phone_os_clean_Unknown', 'phone_os_clean_android',
       'phone_os_clean_apple', 'star_sign_clean_Unknown',
       'star_sign_clean_aquarius', 'star_sign_clean_aries',
       'star_sign_clean_cancer', 'star_sign_clean_capricorn',
       'star_sign_clean_gemini', 'star_sign_clean_leo',
       'star_sign_clean_libra', 'star_sign_clean_pisces',
       'star_sign_clean_sagittarius', 'star_sign_clean_scorpio',
       'star_sign_clean_taurus', 'star_sign_cl

In [12]:
X_train_enc.head()

,height,weight,sleepiness,iq,fb_friends,yt,is_yt_invalid,is_phone_os_invalid,is_weight_missing,is_weight_outlier,is_height_missing,is_height_outlier,is_iq_outlier,is_fb_friends_outlier,is_yt_outlier,is_star_sign_invalid,is_intro_missing,intro_char_length,intro_word_count,intro_is_all_lower,intro_is_all_upper,is_intro_anomalous,self_intro_sim_to_male,self_intro_sim_to_female,self_intro_male_minus_female,phone_os_clean_Unknown,phone_os_clean_android,phone_os_clean_apple,star_sign_clean_Unknown,star_sign_clean_aquarius,star_sign_clean_aries,star_sign_clean_cancer,star_sign_clean_capricorn,star_sign_clean_gemini,star_sign_clean_leo,star_sign_clean_libra,star_sign_clean_pisces,star_sign_clean_sagittarius,star_sign_clean_scorpio,star_sign_clean_taurus,star_sign_clean_virgo
145,168.00,63.00,2.00,100.0,328.0,2.0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0.000000,0.000000,0.000000,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
78,170.00,59.68,3.30,100.0,99.0,10.0,0,0,1,0,0,0,1,0,0,0,0,4,1,1,0,0,0.316101,0.311759,0.004342,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
16,156.00,47.00,2.72,130.0,400.0,3.5,0,0,0,0,0,0,0,0,0,0,0,26,5,0,0,0,0.274845,0.322193,-0.047348,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
149,168.00,49.00,1.00,100.0,2000.0,24.0,0,1,0,0,0,0,1,0,1,0,0,2,1,0,0,0,0.449749,0.474628,-0.024879,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
293,180.58,76.00,4.00,140.0,1287.0,24.0,0,0,0,0,1,0,1,0,1,1,0,17,4,0,0,0,0.250873,0.253375,-0.002502,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0


In [13]:
missing_train = X_train_enc.isnull().sum()
missing_valid = X_valid_enc.isnull().sum()
missing_test = X_test_enc.isnull().sum()

print("訓練集剩餘缺失值 (大於 0 的欄位)：")
print(missing_train[missing_train > 0])
print("\n驗證集剩餘缺失值 (大於 0 的欄位)：")
print(missing_valid[missing_valid > 0])
print("\n測試集剩餘缺失值 (大於 0 的欄位)：")
print(missing_test[missing_test > 0])


訓練集剩餘缺失值 (大於 0 的欄位)：
Series([], dtype: int64)

驗證集剩餘缺失值 (大於 0 的欄位)：
Series([], dtype: int64)

測試集剩餘缺失值 (大於 0 的欄位)：
Series([], dtype: int64)


## Phase 4: 資料重採樣 (ENN)
使用 ENN 清理男女分類邊界上的雜訊樣本

In [ ]:
# print("重採樣前樣本數:", X_train_enc.shape[0])
# print("原比例:\n", y_train.value_counts())

# # 使用 ENN，針對邊界雜訊進行清除 (保留決策邊界清晰的資料)
# enn = EditedNearestNeighbours(n_neighbors=3)
# X_train_resampled, y_train_resampled = enn.fit_resample(X_train_enc, y_train)

# print("重採樣後樣本數:", X_train_resampled.shape[0])
# print("新比例:\n", y_train_resampled.value_counts())

## Phase 5: 特徵選擇與模型訓練 (Embedded Method)
先用 Random Forest 評估特徵重要性，留下最具預測力的核心特徵，隨後進行最終模型訓練。

In [ ]:
# fs_model = RandomForestClassifier(n_estimators=100, random_state=42)
# fs_model.fit(X_train_resampled, y_train_resampled)

# feat_imp = pd.Series(fs_model.feature_importances_, index=X_train_resampled.columns).sort_values(ascending=False)

# plt.figure(figsize=(10,6))
# feat_imp.head(20).plot(kind='bar')
# plt.title("Top 20 Feature Importances")
# plt.show()

# # 篩選重要性 > 0 的核心特徵
# selected_features = feat_imp[feat_imp > 0].index.tolist()
# print(f"從 {X_train_resampled.shape[1]} 個特徵中選出 {len(selected_features)} 個核心特徵。")

# X_train_sel = X_train_resampled[selected_features]
# X_valid_sel = X_valid_enc[selected_features]
# X_test_sel  = X_test_enc[selected_features]

## 最終模型訓練與評估 

In [14]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
# 設定實驗策略 (
resampling_strategies = ['none', 'enn', 'smoteenn']
feature_selection_strategies = [
    'all_features',
    'rf_importance_mean_threshold',
    'l1_based',
    'mean_threshold',
    'top_15_rf',  
    'top_20_rf',
    'top_30_rf'   
]
base_model_name = 'xgboost'

In [15]:
print(f'實驗預計執行 {len(resampling_strategies) * len(feature_selection_strategies)} 組組合...')

# XGBoost 需要連續標籤 (0, 1)
label_offset = y_train.min()
y_train_cv = y_train - label_offset
y_valid_cv = y_valid - label_offset

# 直接使用前面處理好的資料
X_train_base = X_train_enc.copy()
X_valid_base = X_valid_enc.copy()
X_test_base = X_test_enc.copy()

# train CV 設定
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def apply_resampling(strategy, X, y):
    if strategy == 'enn':
        sampler = EditedNearestNeighbours(n_neighbors=3)
    elif strategy == 'smoteenn':
        sampler = SMOTEENN(random_state=42)
    elif strategy == 'none':
        return X.copy(), y.copy()
    else:
        raise ValueError(f'未知重採樣策略: {strategy}')

    X_res, y_res = sampler.fit_resample(X, y)
    return X_res, y_res

# --- 核心函數：特徵選擇 ---
def select_features(strategy, X_res, y_res, X_valid_ref, X_test_ref):
    cols = X_res.columns
    cols_arr = np.array(cols)

    if strategy == 'all_features':
        selected_cols = cols.tolist()

    elif strategy == 'rf_importance_mean_threshold':
        rf_fs = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        rf_fs.fit(X_res, y_res)
        imp = pd.Series(rf_fs.feature_importances_, index=cols)
        threshold = imp.mean()
        selected_cols = imp[imp >= threshold].index.tolist()
        if len(selected_cols) == 0:
            selected_cols = imp.sort_values(ascending=False).head(20).index.tolist()

    elif strategy == 'l1_based':
        l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.5, max_iter=3000, random_state=42)
        l1.fit(X_res, y_res)
        coef_abs = np.abs(l1.coef_).ravel()
        selected_cols = cols_arr[coef_abs > 1e-8].tolist()
        if len(selected_cols) == 0:
            top_idx = np.argsort(coef_abs)[-20:]
            selected_cols = cols_arr[top_idx].tolist()

    elif strategy == 'mean_threshold':
        mi = mutual_info_classif(X_res, y_res, random_state=42)
        mi_s = pd.Series(mi, index=cols)
        threshold = mi_s.mean()
        selected_cols = mi_s[mi_s >= threshold].index.tolist()
        if len(selected_cols) == 0:
            selected_cols = mi_s.sort_values(ascending=False).head(20).index.tolist()

    elif strategy in ['top_15_rf', 'top_20_rf', 'top_30_rf']:
        k = 15 if strategy == 'top_15_rf' else (20 if strategy == 'top_20_rf' else 30)
        rf_fs = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        rf_fs.fit(X_res, y_res)
        imp = pd.Series(rf_fs.feature_importances_, index=cols)
        selected_cols = imp.sort_values(ascending=False).head(k).index.tolist()

    else:
        raise ValueError(f'未知特徵選擇策略: {strategy}')

    # 確保順序與去重
    selected_cols = list(dict.fromkeys(selected_cols))
    print(f"   [{strategy}] 選出 {len(selected_cols)} 特徵。")

    X_res_sel = X_res[selected_cols]
    X_valid_sel = X_valid_ref[selected_cols]
    X_test_sel = X_test_ref[selected_cols]

    return selected_cols, X_res_sel, X_valid_sel, X_test_sel

# --- 開始實驗迴圈 ---
experiment_rows = []
all_artifacts = []
best_artifact = None
best_score = -1
resampling_sample_counts = {}

for rs in resampling_strategies:
    print(f"\n[Resampling] 正在執行 {rs}...")
    X_res, y_res = apply_resampling(rs, X_train_base, y_train_cv)
    resampled_n = int(X_res.shape[0])
    resampling_sample_counts[rs] = resampled_n
    print(f"   -> 重採樣後訓練樣本數: {resampled_n}")

    for fs in feature_selection_strategies:
        print(f"  > [Feature Selection] {fs}", end="... ")

        selected_cols, X_res_sel, X_valid_sel, X_test_sel = select_features(
            fs, X_res, y_res, X_valid_base, X_test_base
        )

        model_params = dict(
            objective='binary:logistic',
            eval_metric='logloss',
            learning_rate=0.05,
            n_estimators=300,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=5,
            random_state=42
        )

        # 先做 train CV=3
        cv_model = XGBClassifier(**model_params)
        cv_scores = cross_val_score(
            cv_model,
            X_res_sel,
            y_res,
            cv=cv3,
            scoring='f1_macro',
            n_jobs=-1
        )
        train_cv3_mean = float(cv_scores.mean())

        # 再用全量重採樣 train 訓練，評估 valid
        model = XGBClassifier(**model_params)
        model.fit(X_res_sel, y_res)

        valid_pred = model.predict(X_valid_sel)
        f1 = f1_score(y_valid_cv, valid_pred, average='macro')

        print(f"完成 (Train CV3 F1: {train_cv3_mean:.4f}, Valid F1: {f1:.4f}, Features: {len(selected_cols)})")

        row = {
            'resampling': rs,
            'feature_selection': fs,
            'n_samples': resampled_n,
            'n_features': len(selected_cols),
            'train_cv3_f1_macro': train_cv3_mean,
            'valid_f1_macro': f1
        }
        experiment_rows.append(row)

        artifact = {
            'model': model,
            'resampling': rs,
            'feature_selection': fs,
            'selected_cols': selected_cols,
            'X_test_sel': X_test_sel,
            'n_samples': resampled_n,
            'train_cv3_f1_macro': train_cv3_mean,
            'valid_f1_macro': f1,
            'n_features': len(selected_cols)
        }
        all_artifacts.append(artifact)

        if f1 > best_score:
            best_score = f1
            best_artifact = artifact

# 產出重採樣樣本數摘要
sampling_summary_df = pd.DataFrame([
    {'resampling': k, 'n_train_samples': v}
    for k, v in resampling_sample_counts.items()
]).sort_values('resampling').reset_index(drop=True)

print('\n' + '='*30)
print('     重採樣策略樣本數摘要')
print('='*30)
print(sampling_summary_df)

# 產出完整報表
experiment_df = pd.DataFrame(experiment_rows).sort_values('valid_f1_macro', ascending=False).reset_index(drop=True)
print('\n' + '='*30)
print('       3x7 網格實驗結果')
print('='*30)
print(experiment_df)

# 取 Top 3
top3_artifacts = sorted(all_artifacts, key=lambda x: x['valid_f1_macro'], reverse=True)[:3]

top3_df = pd.DataFrame([
    {
        'rank': i + 1,
        'resampling': art['resampling'],
        'feature_selection': art['feature_selection'],
        'n_samples': art['n_samples'],
        'n_features': art['n_features'],
        'train_cv3_f1_macro': art['train_cv3_f1_macro'],
        'valid_f1_macro': art['valid_f1_macro']
    }
    for i, art in enumerate(top3_artifacts)
])

print('\n' + '='*30)
print('       Top 3 組合')
print('='*30)
print(top3_df)

# 維持舊變數（Top 1）給後續相容
final_model = top3_artifacts[0]['model']
X_test_best = top3_artifacts[0]['X_test_sel']
best_selected_features = top3_artifacts[0]['selected_cols']

實驗預計執行 21 組組合...

[Resampling] 正在執行 none...
   -> 重採樣後訓練樣本數: 338
  > [Feature Selection] all_features...    [all_features] 選出 41 特徵。
完成 (Train CV3 F1: 0.8348, Valid F1: 0.8599, Features: 41)
  > [Feature Selection] rf_importance_mean_threshold...    [rf_importance_mean_threshold] 選出 10 特徵。
完成 (Train CV3 F1: 0.8312, Valid F1: 0.8599, Features: 10)
  > [Feature Selection] l1_based...    [l1_based] 選出 14 特徵。
完成 (Train CV3 F1: 0.8248, Valid F1: 0.8103, Features: 14)
  > [Feature Selection] mean_threshold...    [mean_threshold] 選出 8 特徵。
完成 (Train CV3 F1: 0.8275, Valid F1: 0.8419, Features: 8)
  > [Feature Selection] top_15_rf...    [top_15_rf] 選出 15 特徵。
完成 (Train CV3 F1: 0.8419, Valid F1: 0.8599, Features: 15)
  > [Feature Selection] top_20_rf...    [top_20_rf] 選出 20 特徵。
完成 (Train CV3 F1: 0.8396, Valid F1: 0.8773, Features: 20)
  > [Feature Selection] top_30_rf...    [top_30_rf] 選出 30 特徵。
完成 (Train CV3 F1: 0.8361, Valid F1: 0.8599, Features: 30)

[Resampling] 正在執行 enn...
   -> 重採樣後訓練樣本數: 24

## 產生 Kaggle Submission 檔案

In [16]:
if test_ids is not None:
    if 'top3_artifacts' in globals() and len(top3_artifacts) > 0:
        for i, art in enumerate(top3_artifacts, start=1):
            preds_cv = art['model'].predict(art['X_test_sel'])
            preds = preds_cv + label_offset

            submission_i = pd.DataFrame({
                'id': test_ids,
                'gender': preds
            })

            out_path = f'submission_top{i}.csv'
            submission_i.to_csv(out_path, index=False)
            print(f'{out_path} 儲存成功')

        print('\nTop1 預覽：')
        print(pd.read_csv('submission_top1.csv').head())
    else:
        # fallback: 僅存一份
        test_preds_cv = final_model.predict(X_test_best)
        test_preds = test_preds_cv + label_offset

        submission = pd.DataFrame({
            'id': test_ids,
            'gender': test_preds
        })

        submission.to_csv('submission2.csv', index=False)
        print('submission2.csv 儲存成功，前幾筆如下：')
        print(submission.head())
else:
    print('Error: Test data id column not found...')


submission_top1.csv 儲存成功
submission_top2.csv 儲存成功
submission_top3.csv 儲存成功

Top1 預覽：
   id  gender
0   1       1
1   2       1
2   3       2
3   4       1
4   5       1
